# 02 — Scenario Generation

The Mean-CVaR optimiser's view of tail risk is shaped entirely by the
scenario panel it sees. This notebook compares the four supported
methods side by side:

| Method | Captures |
|---|---|
| `historical` | observed return rows (i.i.d. resampling) |
| `block` | autocorrelation and volatility clustering |
| `gaussian` | mean / covariance only |
| `student_t` | fat tails (lower df → heavier tails) |

## 1. Synthetic universe (same 10-asset panel)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kurtosis

from benchmarks.base import generate_synthetic_dataset
from services.scenario_generation import ScenarioConfig, generate_scenarios

ds = generate_synthetic_dataset(n_assets=10, n_history=756, seed=42)
n_scenarios = 5000
methods = ["historical", "block", "gaussian", "student_t"]

panels = {
    m: generate_scenarios(
        ds.daily_returns,
        ScenarioConfig(method=m, n_scenarios=n_scenarios,
                       block_size=20, df=4.0, seed=42),
    )
    for m in methods
}
for m, p in panels.items():
    print(f"{m:>11s}: shape={p.shape}  worst={p.min():.4f}  kurt={kurtosis(p[:, 0]):.2f}")

## 2. Tail-loss histograms

Take the equal-weight portfolio (no optimisation noise) and plot the
loss distribution under each method. Heavier-tailed methods stretch
the histogram to the right.

In [ ]:
w_eq = np.full(ds.n_assets, 1.0 / ds.n_assets)

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
for ax, (method, panel) in zip(axes.ravel(), panels.items()):
    losses = -(panel @ w_eq)
    var95 = np.quantile(losses, 0.95)
    cvar95 = losses[losses >= var95].mean()
    ax.hist(losses, bins=80, alpha=0.7)
    ax.axvline(var95, color="orange", linestyle="--", label=f"VaR95={var95:.3f}")
    ax.axvline(cvar95, color="red", linestyle="--", label=f"CVaR95={cvar95:.3f}")
    ax.set_title(f"{method} (n={n_scenarios})")
    ax.legend(loc="upper right", fontsize=8)
plt.suptitle("Loss distribution by scenario method (equal-weight portfolio)")
plt.tight_layout()
plt.show()

## 3. How tail thickness affects Mean-CVaR weights

Re-solve Mean-CVaR with each scenario panel and compare the
resulting CVaR. Heavier-tailed panels push the optimiser to
demand more downside protection.

In [ ]:
from core.portfolio_optimizer import run_optimization

rows = []
for method, panel in panels.items():
    res = run_optimization(
        returns=ds.mu, covariance=ds.Sigma,
        objective="mean_cvar", scenarios=panel,
        weight_min=0.0, weight_max=0.30,
        confidence_level=0.95, risk_aversion=1.0,
    )
    rows.append([method, res.expected_return, res.sharpe_ratio,
                 res.var_95, res.cvar_95, res.solve_time_ms])

header = ["method", "exp_return", "sharpe", "var95", "cvar95", "ms"]
print("  ".join(f"{h:>12}" for h in header))
for r in rows:
    cells = [r[0]] + [f"{v:.4f}" for v in r[1:-1]] + [f"{r[-1]:.1f}"]
    print("  ".join(f"{c:>12}" for c in cells))

## 4. When to use which method

- **`historical`** — Simplest baseline. Sample size limited by the
  observed history; cannot extrapolate beyond observed scenarios.
- **`block`** — Recommended default for daily returns. Preserves
  short-term autocorrelation; `block_size=20` is a sensible start.
- **`gaussian`** — Fast and smooth. Underestimates tail risk in
  real markets — use only for quick development iterations.
- **`student_t`** — Fat tails. Set `df=3` for crypto, `df=4–6` for
  developed-market equities.